# 24 — Graph RAG: Knowledge Graphs + Multi-hop Retrieval
## Entities, Relationships, Multi-Hop — Knowledge Graph Retrieval
⏱ ~15 min

**Graph RAG** is the pattern that fixes a blind spot in standard similarity search: relational questions. When the answer to "Who founded the company that makes LangGraph?" requires connecting facts across two documents, a vector store cannot help — it ranks documents by similarity to the query, not by whether they form a chain of evidence. A knowledge graph can.

By the end of this session you will understand *why* similarity search fails on multi-hop questions, *how* to extract a knowledge graph from text using structured LLM output, and *how* to wire entity lookup + graph traversal into a three-node LangGraph pipeline.

---

### Workshop Roadmap

| # | Topic |
|---|-------|
| 1 | **Concepts** — Vector RAG vs Graph RAG: what breaks and why |
| 2 | **Knowledge Graphs** — Nodes, edges, triples, and NetworkX |
| 3 | **Triple Extraction** — Entity and relationship extraction with structured output |
| 4 | **Graph Traversal** — Finding multi-hop paths from a question |
| 5 | **LangGraph Pipeline** — Three-node graph: find_entities → retrieve_subgraph → generate |
| 6 | **Comparison** — Graph RAG vs vector RAG on the same questions |
| 7 | **Evaluation** — Debugging entity matching and traversal quality |
| ★ | **Extensions** (bonus) — Hybrid Graph+Vector, community detection |

---

### Prerequisites
- Python 3.10+ (a `.venv` with the requirements already installed)
- An `OPENAI_API_KEY` set in `.env` (or Colab Secrets)

### Key References
> Edge, D., Trinh, H., et al. (2024). *From Local to Global: A Graph RAG Approach to Query-Focused Summarization.* Microsoft Research. https://arxiv.org/abs/2404.16130  
> Lewis, P., Perez, E., et al. (2020). *Retrieval-Augmented Generation for Knowledge-Intensive NLP Tasks.* NeurIPS 2020. https://arxiv.org/abs/2005.11401  
> NetworkX documentation: https://networkx.org/documentation/stable/  
> LangGraph documentation: https://langchain-ai.github.io/langgraph/

In [ ]:
# Install dependencies — runs only on Google Colab.
# Local users: your .venv already has everything from requirements.txt.
import sys


def _in_colab():
    try:
        import google.colab

        return True
    except ImportError:
        return False


if _in_colab():
    import subprocess

    subprocess.run(
        [
            sys.executable,
            "-m",
            "pip",
            "install",
            "-q",
            "langchain",
            "langchain-openai",
            "langchain-chroma",
            "langgraph",
            "networkx",
            "matplotlib",
            "python-dotenv",
            "pydantic",
        ]
    )
    print("Colab install complete.")
else:
    print("Local environment detected — skipping install.")

In [ ]:
import os

# ── API key ───────────────────────────────────────────────────────────────────
# Colab: set in Secrets panel (left sidebar key icon)
# Local: create a .env file containing  OPENAI_API_KEY=sk-...
try:
    from google.colab import userdata

    os.environ["OPENAI_API_KEY"] = userdata.get("OPENAI_API_KEY")
except ImportError:
    from dotenv import load_dotenv

    load_dotenv()

# ── Sanity check ──────────────────────────────────────────────────────────────
key = os.environ.get("OPENAI_API_KEY", "")
assert key.startswith("sk-"), (
    "OPENAI_API_KEY not found or looks wrong. "
    "Local: add it to .env  |  Colab: add it in Secrets panel"
)
print(f"OPENAI_API_KEY present: True ({key[:8]}...)")

---

## Part 1 — Why Vector Search Fails on Relational Questions

---

### The Multi-Hop Problem

Standard RAG retrieves documents that are *semantically similar* to the query. That works well when the answer lives inside a single document. It breaks when the answer requires combining facts from multiple documents — what researchers call a **multi-hop** question.

Consider:

> **"Who founded the company that makes LangGraph?"**

```
Document A: "LangGraph is a library built on top of LangChain."
Document B: "LangChain was founded by Harrison Chase in 2022."
```

A query about "LangGraph founder" is semantically close to Document A — but Document A does not contain the answer. Document B has the answer but scores lower on similarity to "LangGraph founder". The retriever picks the wrong document, or picks both and hopes the LLM connects them without context.

---

### Three Retrieval Approaches Compared

| Approach | How it retrieves | Multi-hop support | Cost |
|----------|-----------------|-------------------|------|
| **Vector RAG** | Cosine similarity to query embedding | Poor — single-document focus | Low |
| **Graph RAG** | Entity lookup + edge traversal | Strong — paths span documents | Medium (graph build) |
| **Hybrid** | Vector for candidates + graph for relationships | Best of both | Higher |

---

### Key Vocabulary

| Term | Definition |
|------|------------|
| **Entity** | A named real-world object: person, org, product, place |
| **Relationship / predicate** | A labelled directed link between two entities |
| **Triple** | `(subject, predicate, object)` — the atomic unit of a knowledge graph |
| **Knowledge graph** | A directed graph where nodes are entities and edges are relationships |
| **Multi-hop** | A question whose answer requires traversing 2+ edges in the graph |
| **Subgraph** | The subset of nodes and edges relevant to a specific query |
| **Entity linking** | Mapping query mentions to canonical graph node names |

### Graph RAG Architecture

```
OFFLINE — build once when documents change
──────────────────────────────────────────────────────────────

  Source documents
       │
       ▼
  ┌────────────────────┐
  │  Triple Extractor  │  LLM call per document:
  │  (structured LLM)  │  "LangChain" ──[founded_by]──► "Harrison Chase"
  └─────────┬──────────┘  "LangGraph" ──[built_on]───► "LangChain"
             │
             ▼
  ┌────────────────────┐
  │  NetworkX DiGraph  │  directed graph stored in memory
  │  nodes = entities  │  (persist with pickle or Neo4j for production)
  │  edges = predicates│
  └────────────────────┘


ONLINE — every user query
──────────────────────────────────────────────────────────────

  User: "Who founded the company that makes LangGraph?"
       │
       ▼
  ┌────────────────────┐
  │   find_entities    │  LLM extracts ["LangGraph"] from question,
  │  (structured LLM)  │  matches to canonical graph node names
  └─────────┬──────────┘
             │  entities: ["LangGraph"]
             ▼
  ┌────────────────────┐
  │ retrieve_subgraph  │  graph traversal: collect in/out edges
  │  (pure Python)     │  "LangGraph built_on LangChain"
  │                    │  "LangChain founded_by Harrison Chase"
  └─────────┬──────────┘
             │  context: relational facts as text
             ▼
  ┌────────────────────┐
  │     generate       │  LLM reads ONLY the graph facts
  │       (LLM)        │  → "Harrison Chase"
  └─────────┬──────────┘
             │
             ▼
        Final answer
```

> **Source**: Architecture adapts the GraphRAG pipeline from Edge et al. (2024), Microsoft Research.

---

## Part 2 — Knowledge Graphs with NetworkX

---

### Triples: The Atomic Unit

Every fact in a knowledge graph is a **triple**: `(subject, predicate, object)`.

```
"LangChain was founded by Harrison Chase in 2022."
  │
  ▼  extract
("LangChain",  "founded_by",     "Harrison Chase")
("LangChain",  "founded_in",     "2022")
("LangChain",  "is_a",           "Python framework")
```

In a directed graph:
- **Nodes** = subjects and objects (entities)
- **Edges** = predicates (relationships), always pointing from subject → object

### Entity Types and Relationship Types

| Entity type | Examples |
|-------------|----------|
| **Person** | Harrison Chase, Dario Amodei, Sam Altman |
| **Organisation** | LangChain Inc., Anthropic, OpenAI |
| **Product** | LangGraph, Claude, GPT-4, LangSmith |
| **Date / Number** | 2022, $25M, 2021 |
| **Concept** | "safety", "vector search", "observability" |

| Relationship type | Example |
|------------------|----------|
| **founded_by** | LangChain ──founded_by──► Harrison Chase |
| **built_on** | LangGraph ──built_on──► LangChain |
| **developed_by** | Claude ──developed_by──► Anthropic |
| **left_to_found** | Dario Amodei ──left_to_found──► Anthropic |
| **raised** | LangChain Inc. ──raised──► $25M Series A |

In [ ]:
# ─── 2-A: Build a knowledge graph from hand-crafted triples ──────────────────
# We start manually so you can see exactly what the graph contains
# before we automate extraction with an LLM.

import networkx as nx

G_manual = nx.DiGraph()

# Each tuple is (subject, object, predicate_label)
MANUAL_TRIPLES = [
    ("LangChain", "Harrison Chase", "founded_by"),
    ("LangChain", "2022", "founded_in"),
    ("LangGraph", "LangChain", "built_on"),
    ("LangSmith", "LangChain Inc.", "created_by"),
    ("LangChain Inc.", "Harrison Chase", "led_by"),
    ("Anthropic", "Dario Amodei", "co-founded_by"),
    ("Anthropic", "Daniela Amodei", "co-founded_by"),
    ("Dario Amodei", "OpenAI", "previously_at"),
    ("Claude", "Anthropic", "developed_by"),
    ("GPT-4", "OpenAI", "developed_by"),
    ("Ilya Sutskever", "OpenAI", "co-founded"),
    ("Ilya Sutskever", "SSI", "founded"),
]

for subject, obj, predicate in MANUAL_TRIPLES:
    G_manual.add_edge(subject, obj, predicate=predicate)

print(f"Nodes: {G_manual.number_of_nodes()}")
print(f"Edges: {G_manual.number_of_edges()}")
print()
print("All edges (subject, predicate, object):")
for src, dst, data in G_manual.edges(data=True):
    print(f"  {src:25} ──[{data['predicate']}]──► {dst}")

In [ ]:
# ─── 2-B: Traverse the graph to answer a multi-hop question ──────────────────
# "Who founded the company that makes LangGraph?"
#
# Step 1: find "LangGraph" in the graph
# Step 2: follow LangGraph → LangChain  (built_on)
# Step 3: follow LangChain → Harrison Chase  (founded_by)


def get_neighbors(G: nx.DiGraph, entity: str) -> list[str]:
    """Return all fact strings (in and out edges) for an entity."""
    facts = []
    for _, neighbor, data in G.out_edges(entity, data=True):
        facts.append(f"{entity} {data['predicate']} {neighbor}")
    for source, _, data in G.in_edges(entity, data=True):
        facts.append(f"{source} {data['predicate']} {entity}")
    return facts


print("One-hop from 'LangGraph':")
for f in get_neighbors(G_manual, "LangGraph"):
    print(f"  {f}")

print()
print("One-hop from 'LangChain' (second hop):")
for f in get_neighbors(G_manual, "LangChain"):
    print(f"  {f}")

print()
print("Combined context for the LLM:")
combined = get_neighbors(G_manual, "LangGraph") + get_neighbors(G_manual, "LangChain")
print("\n".join(f"  {f}" for f in combined))
print()
print("→ The answer 'Harrison Chase' is now present in context.")

In [ ]:
# ─── 2-C: Visualize the knowledge graph ──────────────────────────────────────
import matplotlib.pyplot as plt

fig, ax = plt.subplots(figsize=(12, 7))
pos = nx.spring_layout(G_manual, seed=42, k=2)

nx.draw_networkx_nodes(G_manual, pos, node_color="#4a9eff", node_size=1200, ax=ax)
nx.draw_networkx_labels(G_manual, pos, font_size=8, font_color="white", font_weight="bold", ax=ax)
nx.draw_networkx_edges(
    G_manual, pos, arrows=True, arrowsize=15,
    edge_color="#555", connectionstyle="arc3,rad=0.1", ax=ax
)
edge_labels = {(s, d): data["predicate"] for s, d, data in G_manual.edges(data=True)}
nx.draw_networkx_edge_labels(
    G_manual, pos, edge_labels=edge_labels, font_size=7, font_color="#c0392b", ax=ax
)

ax.set_title("Knowledge Graph — AI Ecosystem (hand-crafted triples)", fontsize=13)
ax.axis("off")
plt.tight_layout()
plt.show()

print("Each blue node is an entity. Red labels on arrows are predicates.")
print("Trace: LangGraph → LangChain → Harrison Chase (two hops, two arrows).")

---

## Part 3 — Triple Extraction with a Structured LLM

---

Building the knowledge graph by hand works for demos. In production you extract triples automatically from raw text using an LLM with **structured output** — the model returns a validated Pydantic schema rather than free text.

### Why Structured Output?

```
Unstructured output (bad):
  "The triples are: LangChain was founded by Harrison Chase, and
   LangGraph is built on LangChain."
  → requires fragile parsing to extract entities

Structured output (good):
  [
    {"subject": "LangChain",  "predicate": "founded_by",  "object": "Harrison Chase"},
    {"subject": "LangGraph",  "predicate": "built_on",    "object": "LangChain"}
  ]
  → directly usable as graph.add_edge(subject, object, predicate=predicate)
```

LangChain's `.with_structured_output(Schema)` wraps the model call and guarantees the response parses into the given Pydantic class.

### Triple Extraction Quality Tips

| Problem | Symptom | Fix |
|---------|---------|-----|
| Verbose entity names | `"the LangChain framework"` not `"LangChain"` | Prompt: "use short canonical names" |
| Hallucinated triples | Facts not in source text | Prompt: "only triples clearly stated in the text" |
| Inconsistent predicates | `founded_by` vs `was_founded_by` | Normalise to snake_case in post-processing |
| Missing entities | Key facts skipped | Increase `temperature`, use a stronger model |

In [ ]:
# ─── 3-A: Define the Pydantic schema and extractor ───────────────────────────
from pydantic import BaseModel
from langchain_core.messages import HumanMessage, SystemMessage
from langchain_openai import ChatOpenAI


class Triple(BaseModel):
    subject: str
    predicate: str
    object: str


class Triples(BaseModel):
    triples: list[Triple]


# temperature=0 — deterministic extraction; hallucinated triples corrupt the graph permanently
llm = ChatOpenAI(model="gpt-5-nano", temperature=0)
# with_structured_output forces the model to return a valid Pydantic object;
# the schema acts as the prompt — field names and docstrings become the LLM's instructions
extractor = llm.with_structured_output(Triples)


def extract_triples(doc: str) -> list[Triple]:
    result = extractor.invoke(
        [
            SystemMessage(
                "Extract factual (subject, predicate, object) triples from the text. "
                "Use short canonical entity names (e.g. 'LangChain', not 'the LangChain framework'). "
                "Predicates must be snake_case (e.g. 'founded_by', 'built_on'). "
                "Return ONLY triples clearly stated in the text — do not infer."
            ),
            HumanMessage(doc),
        ]
    )
    return result.triples


# Test on a single sentence
sample = "LangChain was founded by Harrison Chase in 2022 as an open-source Python framework."
triples = extract_triples(sample)
print(f"Source: '{sample}'")
print(f"Extracted {len(triples)} triple(s):")
for t in triples:
    print(f"  ({t.subject!r}, {t.predicate!r}, {t.object!r})")

In [ ]:
# ─── 3-B: Build the full knowledge graph from a document corpus ───────────────
# This is the offline phase: one LLM call per document.
# Once built, all retrieval is pure graph traversal — no LLM calls until generate.

DOCS = [
    "LangChain was founded by Harrison Chase in 2022 as an open-source Python framework for LLM applications.",
    "LangGraph is a library built on top of LangChain by Harrison Chase, adding stateful graph-based agent workflows.",
    "LangSmith is an observability platform created by LangChain Inc. for tracing and evaluating LLM pipelines.",
    "Anthropic was founded by Dario Amodei and Daniela Amodei in 2021 after they left OpenAI.",
    "Claude is the AI assistant developed by Anthropic, designed with a focus on safety and helpfulness.",
    "OpenAI was co-founded by Sam Altman, Elon Musk, Greg Brockman, and Ilya Sutskever in 2015.",
    "GPT-4 is a large language model developed by OpenAI, released in March 2023.",
    "Ilya Sutskever co-founded OpenAI and later founded Safe Superintelligence Inc. (SSI) in 2024.",
    "Dario Amodei previously served as VP of Research at OpenAI before co-founding Anthropic.",
    "LangChain Inc. raised a $25M Series A in 2023 and later a $35M round led by Sequoia Capital.",
]


# Offline build: each doc costs one LLM call; the resulting graph serves all future queries for free.
# Trade-off: graph quality = extraction quality — bad prompts produce noisy edges that hurt retrieval
def build_graph(docs: list[str]) -> nx.DiGraph:
    G = nx.DiGraph()
    print(f"Extracting triples from {len(docs)} documents...")
    for i, doc in enumerate(docs):
        triples = extract_triples(doc)
        for t in triples:
            G.add_edge(t.subject, t.object, predicate=t.predicate, source_doc=i)
        print(f"  [{i + 1:2d}/{len(docs)}] {len(triples)} triple(s): {doc[:60]}...")
    return G


G = build_graph(DOCS)
print()
print(f"Graph summary: {G.number_of_nodes()} nodes, {G.number_of_edges()} edges")
print(f"Nodes: {sorted(G.nodes())}")

In [ ]:
# ─── 3-C: Inspect the extracted graph ────────────────────────────────────────
# A healthy graph extraction produces consistent entity names and predicates.
# Look for: duplicate nodes with different capitalisation, overly verbose names.

print("All edges in the extracted knowledge graph:")
print(f"{'Subject':30} {'Predicate':25} {'Object'}")
print("-" * 75)
for src, dst, data in sorted(G.edges(data=True)):
    print(f"{src:30} {data['predicate']:25} {dst}")

print()
print("Degree centrality (most connected nodes):")
centrality = nx.degree_centrality(G)
top_nodes = sorted(centrality.items(), key=lambda x: -x[1])[:5]
for node, score in top_nodes:
    print(f"  {node:30} centrality={score:.3f}")

---

## Part 4 — Graph Traversal Strategies

---

Once the graph is built, retrieval is a traversal problem. Given a set of entities mentioned in the question, you want to collect the surrounding facts and pass them as context to the LLM.

### Traversal Strategies Compared

| Strategy | What it collects | Hops | Good for |
|----------|-----------------|------|----------|
| **1-hop** | Direct in/out edges of matched entity | 1 | Simple lookup questions |
| **2-hop** | Edges of matched entity + their neighbours | 2 | Multi-hop "who made X?" questions |
| **BFS k-hop** | All nodes within k steps | k | Broad context, risk of noise |
| **Shortest path** | Minimum path between two entities | varies | "What connects A to B?" |
| **Subgraph** | All nodes reachable from entity set | full | Community detection / summarization |

For this workshop we use **1-hop** traversal (collect all in/out edges of each matched entity). The LLM's entity-matching step finds the right entry points; the 1-hop collection often surfaces enough context for 2-hop questions because the graph was built from related documents.

In [ ]:
# ─── 4-A: Compare traversal strategies ───────────────────────────────────────


# 1-hop is cheap and precise; 2-hop answers bridging questions but doubles context size and noise.
# Most production systems default to 1-hop and only widen when the LLM signals missing context.
def collect_1hop(G: nx.DiGraph, entity: str) -> list[str]:
    """All direct in/out edges from a single node."""
    facts = []
    for _, neighbor, data in G.out_edges(entity, data=True):
        facts.append(f"{entity} {data['predicate']} {neighbor}")
    for source, _, data in G.in_edges(entity, data=True):
        facts.append(f"{source} {data['predicate']} {entity}")
    return facts


def collect_2hop(G: nx.DiGraph, entity: str) -> list[str]:
    """1-hop neighbours + their own 1-hop edges (2 hops total)."""
    facts = collect_1hop(G, entity)
    seen = set(facts)
    for _, neighbor in G.out_edges(entity):
        for f in collect_1hop(G, neighbor):
            if f not in seen:
                facts.append(f)
                seen.add(f)
    return facts


entity = "LangGraph"
print(f"Entity: '{entity}'")
print()

one_hop = collect_1hop(G, entity)
print(f"1-hop ({len(one_hop)} facts):")
for f in one_hop:
    print(f"  {f}")

print()
two_hop = collect_2hop(G, entity)
print(f"2-hop ({len(two_hop)} facts):")
for f in two_hop:
    print(f"  {f}")

print()
print("Note: 2-hop surfaces 'LangChain founded_by Harrison Chase'")
print("which is the answer to 'Who founded the company that makes LangGraph?'")

In [ ]:
# ─── 4-B: Entity linking — match question mentions to graph nodes ─────────────
# The user might write "Ilya" but the graph has "Ilya Sutskever".
# We use case-insensitive partial matching (cheap) before calling the LLM matcher.


# Substring match before LLM matching saves latency and cost.
# Alternative: embed all node names and use cosine similarity for richer fuzzy matching.
def fuzzy_match_entity(G: nx.DiGraph, mention: str, top_k: int = 3) -> list[str]:
    """Find graph nodes whose name contains `mention` or vice versa."""
    mention_lower = mention.lower()
    matches = [
        n
        for n in G.nodes()
        if mention_lower in n.lower() or n.lower() in mention_lower
    ]
    return matches[:top_k]


test_mentions = ["Ilya", "LangGraph", "OpenAI", "claude", "Dario", "NotInGraph"]

print("Fuzzy entity matching:")
print(f"{'Mention':20} {'Matched node(s)'}")
print("-" * 50)
for mention in test_mentions:
    matched = fuzzy_match_entity(G, mention)
    print(f"{mention:20} {matched if matched else '(no match)'}")

---

## Part 5 — The Three-Node LangGraph Pipeline

---

The Graph RAG pipeline is a **StateGraph** with exactly three nodes. State flows through the graph as a `TypedDict`.

```
START
  │
  ▼
find_entities    ← LLM call: extract named entities from question,
  │                           match to canonical graph node names
  │  state.entities = ["LangGraph"]
  ▼
retrieve_subgraph ← Pure Python: collect in/out edges for matched entities
  │                              (no LLM call — fast, deterministic)
  │  state.context = "LangGraph built_on LangChain\nLangChain founded_by..."
  ▼
generate         ← LLM call: answer from graph facts only
  │
  ▼
END
```

### LLM Calls: Only 2 Per Query

| Node | LLM? | Why |
|------|------|-----|
| `find_entities` | Yes (structured) | Natural language → entity names |
| `retrieve_subgraph` | **No** | Pure graph traversal — zero cost |
| `generate` | Yes | Graph facts → natural language answer |

In [ ]:
# ─── 5-A: Define state and the three node functions ───────────────────────────
from typing import TypedDict
from langgraph.graph import StateGraph, START, END

# TOP_K caps how many graph nodes we retrieve per entity — higher k = more context, higher token cost
TOP_K = 3  # max matched nodes per entity


# TypedDict gives LangGraph typed access to state keys without runtime overhead;
# every node receives and returns a dict subset — keys not returned stay unchanged
class GraphRAGState(TypedDict):
    question: str
    entities: list       # list[str] — canonical entity names extracted from question
    context: str         # relational facts assembled from graph traversal
    answer: str


class EntityMatch(BaseModel):
    entities: list[str]


def find_entities(state: GraphRAGState) -> dict:
    """LLM step: extract entities from the question and match to graph nodes."""
    matcher = llm.with_structured_output(EntityMatch)
    result = matcher.invoke(
        [
            SystemMessage(
                f"Extract the key named entities from this question. "
                f"Match them to canonical names in this list where possible: {sorted(G.nodes())}. "
                "Return only entity names — no explanations."
            ),
            HumanMessage(state["question"]),
        ]
    )
    return {"entities": result.entities}


def retrieve_subgraph(state: GraphRAGState) -> dict:
    """Pure-Python step: collect graph facts for all matched entities."""
    facts, seen = [], set()
    for entity in state["entities"]:
        matches = fuzzy_match_entity(G, entity, top_k=TOP_K)
        for node in matches:
            for f in collect_1hop(G, node):
                if f not in seen:
                    facts.append(f)
                    seen.add(f)
    context = "\n".join(facts) if facts else "No relevant graph context found."
    return {"context": context}


def generate(state: GraphRAGState) -> dict:
    """LLM step: answer from graph facts only."""
    response = llm.invoke(
        [
            SystemMessage(
                "Answer the question using ONLY the knowledge graph facts below. "
                "If the answer cannot be derived from these facts, say so clearly.\n\n"
                f"Graph facts:\n{state['context']}"
            ),
            HumanMessage(state["question"]),
        ]
    )
    return {"answer": response.content}


print("Node functions defined: find_entities, retrieve_subgraph, generate")

In [ ]:
# ─── 5-B: Compile the LangGraph pipeline ─────────────────────────────────────

# compile() locks the graph topology and validates all edges;
# without it the graph is just a builder object — invoke() won't work
builder = StateGraph(GraphRAGState)
builder.add_node("find_entities", find_entities)
builder.add_node("retrieve_subgraph", retrieve_subgraph)
builder.add_node("generate", generate)
builder.add_edge(START, "find_entities")
builder.add_edge("find_entities", "retrieve_subgraph")
builder.add_edge("retrieve_subgraph", "generate")
builder.add_edge("generate", END)

app = builder.compile()
print("Graph RAG pipeline compiled successfully.")

In [ ]:
# ─── 5-C: Run sample queries and inspect the full state trace ────────────────

SAMPLE_QUESTIONS = [
    "Who founded the company that makes LangGraph?",           # multi-hop
    "What did Ilya Sutskever do after leaving OpenAI?",        # direct
    "What observability tool does LangChain Inc. make?",       # direct
    "What company did Dario Amodei leave to found Anthropic?", # multi-hop
]

initial_state = {"question": "", "entities": [], "context": "", "answer": ""}

for question in SAMPLE_QUESTIONS:
    print("=" * 65)
    result = app.invoke({**initial_state, "question": question})
    print(f"Q: {question}")
    print(f"   Entities matched: {result['entities']}")
    n_facts = result['context'].count('\n') + 1 if result['context'].strip() else 0
    print(f"   Context ({n_facts} facts):")
    for line in result["context"].splitlines():
        print(f"     {line}")
    print(f"   A: {result['answer']}")
    print()

---

## Part 6 — Graph RAG vs Vector RAG: Head-to-Head

---

The strongest case for Graph RAG is the multi-hop question. Below we build a vector RAG pipeline over the same documents and run the same questions through both systems. You should observe:

- **Single-hop questions**: vector RAG performs comparably — the answer lives in one document.
- **Multi-hop questions**: graph RAG wins — vector RAG retrieves only one relevant document and the LLM cannot connect the path.

### What Each System Sees for "Who founded the company that makes LangGraph?"

```
Vector RAG retrieves:
  Doc 1: "LangGraph is a library built on top of LangChain..."  ← semantically close
  Doc 2: "LangSmith is an observability platform..."            ← not relevant
  Doc 3: "LangChain Inc. raised a $25M Series A..."            ← not relevant
  → "Harrison Chase" is NOT in the top-3 retrieved chunks

Graph RAG retrieves:
  LangGraph built_on LangChain
  LangChain founded_by Harrison Chase
  LangChain founded_in 2022
  → Answer: "Harrison Chase" ✓
```

In [ ]:
# ─── 6-A: Build a vector RAG baseline over the same DOCS ─────────────────────
from langchain_core.documents import Document
from langchain_chroma import Chroma
from langchain_openai import OpenAIEmbeddings
from langchain_core.output_parsers import StrOutputParser
from langchain_core.prompts import ChatPromptTemplate
from langchain_core.runnables import RunnablePassthrough

embeddings = OpenAIEmbeddings(model="text-embedding-3-small")

vector_store = Chroma.from_documents(
    documents=[Document(page_content=doc, metadata={"idx": i}) for i, doc in enumerate(DOCS)],
    embedding=embeddings,
)

# k=3 controls recall vs precision: lower k = faster, fewer distractors; higher k = more coverage
vector_retriever = vector_store.as_retriever(search_kwargs={"k": 3})

vector_prompt = ChatPromptTemplate.from_messages(
    [
        (
            "system",
            "Answer the question using ONLY the context below. "
            "If the answer is not in the context, say 'I cannot find that in the documents.'\n\n"
            "Context:\n{context}",
        ),
        ("human", "{question}"),
    ]
)


def format_docs(docs):
    return "\n\n".join(d.page_content for d in docs)


vector_chain = (
    {"context": vector_retriever | format_docs, "question": RunnablePassthrough()}
    | vector_prompt
    | llm
# StrOutputParser strips the AIMessage wrapper and returns a plain string;
# without it you'd get an AIMessage object — fine for chaining, awkward for print()
    | StrOutputParser()
)

print(f"Vector store ready: {vector_store._collection.count()} documents")

In [ ]:
# ─── 6-B: Head-to-head comparison ────────────────────────────────────────────

comparison_questions = [
    ("Who founded the company that makes LangGraph?",       "Harrison Chase"),  # multi-hop
    ("What did Ilya Sutskever found in 2024?",               "SSI"),             # direct
    ("What company did Dario Amodei work at before Anthropic?", "OpenAI"),       # direct
]

for question, expected in comparison_questions:
    vector_answer = vector_chain.invoke(question)
    graph_result = app.invoke({**initial_state, "question": question})
    graph_answer = graph_result["answer"]

    print(f"Q:        {question}")
    print(f"Expected: {expected}")
    print(f"Vector:   {vector_answer[:120].replace(chr(10), ' ')}")
    print(f"Graph:    {graph_answer[:120].replace(chr(10), ' ')}")
    print()

print("Observation: multi-hop questions favour Graph RAG; single-hop questions are comparable.")

---

## Part 7 — Debugging and Evaluation

---

### Common Graph RAG Failure Modes

| Failure | Symptom | Root cause | Fix |
|---------|---------|-----------|-----|
| **Entity not found** | `entities=[]` or wrong matches | Mention != canonical name | Improve entity linking; LLM prompt with node list |
| **Empty context** | `"No relevant graph context found."` | Fuzzy match failed | Widen match threshold; add synonym mapping |
| **Wrong entity matched** | Right names, wrong answer | Ambiguous partial match | Prefer longer, more specific matches |
| **Verbose entity names** | `"the LangChain framework"` in graph | Extraction prompt too lenient | Enforce short canonical names in extraction prompt |
| **Hallucination** | LLM invents path not in graph | Model ignores grounding | Stronger system prompt; lower `temperature` |
| **Missing triples** | Key edge not extracted | Model skips implicit facts | More specific extraction prompt; higher `temperature` |

### Debugging Checklist

1. Print `result['entities']` — are the right entities found?
2. Print `result['context']` — is the relevant fact present?
3. Run `collect_1hop(G, entity)` directly — what does the graph contain?
4. Check `G.nodes()` — is the expected entity in the graph at all?
5. Print the full prompt — is the LLM receiving the context you expect?

In [ ]:
# ─── 7-A: Step-by-step diagnostic for any query ──────────────────────────────


def diagnose(question: str):
    blank = {"question": question, "entities": [], "context": "", "answer": ""}
    print(f"DIAGNOSE: '{question}'")
    print()

    # Step 1: entity extraction
    s1 = find_entities(blank)
    print(f"Step 1 — Entity extraction:  {s1['entities']}")

    # Step 2: fuzzy matching to graph nodes
    all_matches = {e: fuzzy_match_entity(G, e) for e in s1["entities"]}
    print(f"Step 2 — Graph node matches: {all_matches}")

    # Step 3: context collected
    s2 = retrieve_subgraph({**blank, **s1})
    n_facts = s2["context"].count("\n") + 1 if s2["context"].strip() else 0
    print(f"Step 3 — Context ({n_facts} facts):")
    for line in s2["context"].splitlines():
        print(f"    {line}")

    # Step 4: final answer
    s3 = generate({**blank, **s1, **s2})
    print(f"Step 4 — Answer: {s3['answer']}")


diagnose("Who founded the company that makes LangGraph?")

In [ ]:
# ─── 7-B: Graph health statistics ────────────────────────────────────────────

print("Knowledge Graph Health Report")
print("=" * 40)
print(f"Nodes:              {G.number_of_nodes()}")
print(f"Edges:              {G.number_of_edges()}")
print(f"Is DAG:             {nx.is_directed_acyclic_graph(G)}")
print(f"Weakly connected:   {nx.number_weakly_connected_components(G)} component(s)")

isolated = [n for n in G.nodes() if G.degree(n) == 0]
print(f"Isolated nodes:     {isolated if isolated else 'none'}")

predicates = sorted(set(data["predicate"] for _, _, data in G.edges(data=True)))
print(f"Unique predicates:  {len(predicates)}")
for p in predicates:
    count = sum(1 for _, _, d in G.edges(data=True) if d["predicate"] == p)
    print(f"  {p:35} ({count} edge{'s' if count != 1 else ''})")

---

## Part 8 ★ — Exercises

---

Work through these in order. Each builds on the previous.

**Exercise 1 — Extend the knowledge base**  
Add two new documents to `DOCS` (e.g. `"Mistral AI was founded by Arthur Mensch in 2023."` and one more of your choice). Re-build the graph with `build_graph()`. Ask the system a question that requires the new facts.

**Exercise 2 — Two-hop retrieval**  
The `retrieve_subgraph` node currently uses `collect_1hop`. Replace it with `collect_2hop` (already defined in Part 4). Re-run the multi-hop questions. Do you get more context? Does the answer improve or stay the same?

**Exercise 3 — Visualize the extracted graph**  
Run `matplotlib` on the LLM-extracted graph `G` (same code pattern as cell 2-C above). Identify any unexpected node names (verbose, duplicate, or misformatted). Try improving the extraction prompt to fix them.

**Exercise 4 — Your own domain**  
Write 8-10 sentences about any domain you know well (film, music, science, sports). Build a graph from those sentences. Run 3 questions: 1 single-hop, 1 multi-hop, 1 out-of-scope. Report what happens in each case.

In [ ]:
# ── Exercise 1 Starter — Extend the knowledge base ───────────────────────────
MY_EXTRA_DOCS = [
    # TODO: add at least 2 new factual sentences about any AI company or product
    "Mistral AI was founded by Arthur Mensch in 2023.",
    # "TODO: add another sentence here.",
]

EXTENDED_DOCS = DOCS + MY_EXTRA_DOCS

# TODO: rebuild the graph
# G_extended = build_graph(EXTENDED_DOCS)

# TODO: compile a new pipeline using G_extended and run a new question
# result = ...
# print(result["answer"])

In [ ]:
# ── Exercise 2 Starter — Two-hop retrieval ────────────────────────────────────
# Replace collect_1hop with collect_2hop in retrieve_subgraph.

def retrieve_subgraph_2hop(state: GraphRAGState) -> dict:
    # TODO: same as retrieve_subgraph but call collect_2hop instead
    facts, seen = [], set()
    for entity in state["entities"]:
        matches = fuzzy_match_entity(G, entity, top_k=TOP_K)
        for node in matches:
            for f in collect_2hop(G, node):  # <── 2-hop here
                if f not in seen:
                    facts.append(f)
                    seen.add(f)
    context = "\n".join(facts) if facts else "No relevant graph context found."
    return {"context": context}


# TODO: build a new StateGraph using retrieve_subgraph_2hop, compile it,
# then re-run the multi-hop questions and compare context length and answers.

In [ ]:
# ── Exercise 4 Starter — Your own domain ─────────────────────────────────────
MY_DOMAIN_DOCS = [
    # TODO: replace with 8-10 sentences about a domain you know well.
    # Keep sentences self-contained with clear subject-predicate-object structure.
    "The Beatles were formed in Liverpool in 1960.",
    "John Lennon co-founded The Beatles and released a solo career after the band split.",
    # Add more...
]

MY_QUESTIONS = [
    "TODO: a question with a single-hop answer",
    "TODO: a question requiring two hops",
    "TODO: a question the graph cannot answer",
]

# TODO: G_domain = build_graph(MY_DOMAIN_DOCS)
# TODO: compile pipeline with G_domain
# TODO: run the three questions and compare answers

---

## Part 9 ★ — Extensions (Bonus)

---

### Hybrid Graph + Vector RAG

The strongest production systems combine both:

```
Query
  │
  ├── Vector retriever ──► top-k document chunks (semantic match)
  │
  └── Graph traversal ──► entity subgraph facts (relational match)
          │
          ▼
     Merged context → LLM → Answer
```

Vector RAG handles "what does the document say about X?"  
Graph RAG handles "how is X related to Y?"  
Together they cover both semantic and relational retrieval.

### Community Detection for Summarization

Microsoft's GraphRAG (Edge et al. 2024) adds a **community detection** phase (Louvain algorithm) that groups densely-connected entities into communities. Each community is summarized by an LLM. Global queries use these community summaries rather than entity-level traversal — enabling "tell me about the AI ecosystem" style answers.

```python
import networkx.algorithms.community as nx_comm
communities = nx_comm.louvain_communities(G.to_undirected(), seed=42)
for i, community in enumerate(communities):
    print(f"Community {i}: {sorted(community)}")
```

### Persistent Graph Storage

For production, use Neo4j instead of NetworkX:
- NetworkX: in-memory, lost on restart, fine for < 100k nodes
- Neo4j: persistent, Cypher query language, scales to billions of nodes

```python
from langchain_community.graphs import Neo4jGraph
graph = Neo4jGraph(url="bolt://localhost:7687", username="neo4j", password="...")
graph.query("CREATE (n:Entity {name: $name})", {"name": "LangChain"})
```

In [ ]:
# ─── 9-A: Hybrid Graph + Vector context merger ───────────────────────────────


def hybrid_query(question: str, k: int = 2) -> dict:
    """Merge vector-retrieved document chunks with graph-retrieved facts."""
    # Vector context
    vector_docs = vector_retriever.invoke(question)
    vector_context = "\n\n".join(d.page_content for d in vector_docs[:k])

    # Graph context
    graph_result = app.invoke({**initial_state, "question": question})
    graph_context = graph_result["context"]

    # Merge
    merged_context = (
        "=== Document Chunks (semantic match) ===\n"
        + vector_context
        + "\n\n=== Knowledge Graph Facts (relational match) ===\n"
        + graph_context
    )

    answer = llm.invoke(
        [
            SystemMessage(
                "Answer the question using the context below. "
                "Prefer specific facts from the Knowledge Graph section for relational questions.\n\n"
                + merged_context
            ),
            HumanMessage(question),
        ]
    ).content

    return {"answer": answer, "vector_context": vector_context, "graph_context": graph_context}


q = "Who founded the company that makes LangGraph?"
result = hybrid_query(q)
print(f"Q: {q}")
print(f"Hybrid answer: {result['answer']}")

In [ ]:
# ─── 9-B: Community detection (Louvain) ──────────────────────────────────────
# Finds densely-connected clusters of entities — the foundation of Microsoft's
# global GraphRAG summarization approach (Edge et al. 2024).

import networkx.algorithms.community as nx_comm

G_undirected = G.to_undirected()
# seed=42 makes Louvain deterministic; the algorithm is otherwise stochastic
communities = list(nx_comm.louvain_communities(G_undirected, seed=42))

print(f"Detected {len(communities)} communities:\n")
for i, community in enumerate(communities):
    print(f"  Community {i} ({len(community)} nodes): {sorted(community)}")

print()
print("Interpretation: nodes in the same community are densely interconnected.")
print("Each community could be summarized separately for global 'tell me about...' queries.")

---

## What's Next?

You now have a complete Graph RAG pipeline. Here is where to go deeper:

### Related examples in this repo
- **22-hybrid-search-rag** ([`../22-hybrid-search-rag/`](../22-hybrid-search-rag/)): BM25 + vector for single-document factual retrieval — pair it with this example for a full hybrid system.
- **13-structured-output** ([`../13-structured-output/`](../13-structured-output/)): deeper dive into the `with_structured_output` + Pydantic pattern used here for triple extraction.
- **25-adaptive-rag** ([`../25-adaptive-rag/`](../25-adaptive-rag/)): classify queries first, then route to vector RAG, graph RAG, or direct answer — the natural evolution of this notebook.

### Scale the graph
- **Neo4j**: replace NetworkX with `langchain_community.graphs.Neo4jGraph` for persistent, production-grade storage with Cypher query support.
- **Batch extraction**: run triple extraction in parallel with `asyncio.gather()` — the bottleneck is LLM calls, not graph operations.
- **Incremental updates**: `G.add_edge()` works on a live graph — add new documents without rebuilding from scratch.

### Improve retrieval quality
- **Entity normalisation**: canonical name resolution (e.g. `"Sam Altman"` == `"Samuel Altman"`) before inserting into the graph.
- **Predicate normalisation**: map `"was_founded_by"` → `"founded_by"` for consistent traversal.
- **Confidence scoring**: ask the extractor to include a confidence score per triple; filter low-confidence edges.

### Further reading
- Edge, D., et al. (2024). *From Local to Global: A Graph RAG Approach to Query-Focused Summarization.* Microsoft. https://arxiv.org/abs/2404.16130  
- Pan, S., et al. (2024). *Unifying Large Language Models and Knowledge Graphs: A Roadmap.* https://arxiv.org/abs/2306.08302  
- Lewis, P., et al. (2020). *Retrieval-Augmented Generation for Knowledge-Intensive NLP Tasks.* NeurIPS. https://arxiv.org/abs/2005.11401

---

*Workshop complete. The next step is example 25 — adaptive routing that chooses between vector and graph retrieval based on query type.*

---

## Exercise Answer Key

Attempt the exercises before reading these. These are sample solutions, not the only correct approach.

---

### Exercise 1 — Extend the knowledge base

The key step is calling `build_graph(EXTENDED_DOCS)` to re-extract triples and **rebuild the graph from scratch**. You cannot append to the graph after the fact unless you also re-extract triples from the new documents.

A working question: `"Who founded Mistral AI?"` — the answer `"Arthur Mensch"` should appear in context.

**What good output looks like:**
- `entities = ['Mistral AI']`
- `context` contains `"Mistral AI founded_by Arthur Mensch"`
- `answer` mentions `"Arthur Mensch"`

In [ ]:
# Sample answer — Exercise 1
# Uncomment and run. Note: ~12 LLM calls to rebuild the graph.

# EXTENDED_DOCS = DOCS + [
#     "Mistral AI was founded by Arthur Mensch in 2023.",
#     "Mistral AI released the Mixtral model, a mixture-of-experts architecture.",
# ]
#
# G_ext = build_graph(EXTENDED_DOCS)
#
# # Rebuild node functions that close over G_ext
# def find_entities_ext(state):
#     matcher = llm.with_structured_output(EntityMatch)
#     result = matcher.invoke([
#         SystemMessage(f"Extract entities. Match to: {sorted(G_ext.nodes())}."),
#         HumanMessage(state["question"]),
#     ])
#     return {"entities": result.entities}
#
# def retrieve_ext(state):
#     facts, seen = [], set()
#     for entity in state["entities"]:
#         for node in fuzzy_match_entity(G_ext, entity):
#             for f in collect_1hop(G_ext, node):
#                 if f not in seen:
#                     facts.append(f); seen.add(f)
#     return {"context": "\n".join(facts) or "No context found."}
#
# b = StateGraph(GraphRAGState)
# b.add_node("find_entities", find_entities_ext)
# b.add_node("retrieve_subgraph", retrieve_ext)
# b.add_node("generate", generate)
# b.add_edge(START, "find_entities"); b.add_edge("find_entities", "retrieve_subgraph")
# b.add_edge("retrieve_subgraph", "generate"); b.add_edge("generate", END)
# app_ext = b.compile()
#
# r = app_ext.invoke({**initial_state, "question": "Who founded Mistral AI?"})
# print(r["answer"])
print("Uncomment the block above to run Exercise 1 sample solution.")

### Exercise 2 — Two-hop retrieval

Switching to `collect_2hop` increases context size but can introduce noise for simple questions.

On the multi-hop question `"Who founded the company that makes LangGraph?"`, 2-hop traversal starting from `"LangGraph"` will:
- 1st hop: `LangGraph built_on LangChain`
- 2nd hop (from `LangChain`): `LangChain founded_by Harrison Chase`, `LangChain founded_in 2022`, ...

This surfaces `Harrison Chase` more reliably. The trade-off is more context tokens per query and possible irrelevant facts from distant nodes.

**Rule of thumb:** Start with 1-hop. Switch to 2-hop only when entity-to-answer paths are known to span 2 edges.

### Exercise 4 — Your own domain

**What to look for:**
- **Single-hop**: context should contain exactly one directly-relevant fact. Answer should match it precisely.
- **Multi-hop**: requires chaining two edges. If the graph contains both edges, context will surface both facts and the answer will be correct.
- **Out-of-scope**: context will be empty or tangential. The model should respond `"I cannot find that in the graph facts."` If it doesn't, strengthen the system prompt in the `generate` node.

**Common issues:**
- Entity names get split (`"The Beatles"` vs `"Beatles"`) — normalise in the extraction prompt.
- Dates and numbers create isolated noise nodes. Filter them: `G.remove_nodes_from([n for n in G.nodes() if n.isdigit()])`.